# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [29]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [30]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [31]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [32]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [33]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [34]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

Metadata allows the user to 1.) better understand what is being loaded, and 2.) allow  the user to query results based on metadata. This is important for potentially creating more effective and effecient searches. 

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [35]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 135 chunks.
Chunk size: 1000 characters
Chunk overlap: 200 characters


In [36]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

When determining the appropriate chunk size. there are several trade offs: 
1. If the chunk size is too large, you may dilute the information, and potentially dismiss the relevant parts for your query. 
2. If the chunk size is too small, you risk splitting the relevant portions and the chunks can lack context. 

When determining the appropriate overlap, the trade offs are:
1. If the overlap is too large, you reduce the chance of splitting key ideas across two chunks, but this increases redundancy in results and increases the number of chunks to be stored. 
2. If the overlap is too small, you risk losing context at the boundaries of chunks, but the retrieval becomes more effecient. 

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [37]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [38]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [39]:
retrieval_k = 4
retrieval_query =  "What preventive care is recommended for cats?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.657 | page=15 | start_index=1549
17,120,121 This approach will eliminate existing infections, as well as decrease the risk of further infestation and subsequent associated clinical problems. Canine and feline housemates may be at risk of transmission of infectious parasites including roundworm and ﬂeas and therefore should be treated in synchronicity with newly acquired kittens or
--------------------------------------------------------------------------------
Source 2 | score=0.648 | page=2 | start_index=821
alized risk assessment, preventive healthcare strategies, and treat- ment pathways that evolve as the cat matures. An evidence-guided framework for managing a cat ’s healthcare throughout its lifetime has never been more important in feline practice than it is now. Cats are the most popular pet in the United States. 1 A great anomaly in feline prac
--------------------------------------------------------------------------------
Source 3 | score=0.627 | page=19 |

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

The similarity score helps yo uunderstand how close to vectors are in embedding space, usually indicating that a query and chunk share similar language or concepts. 

It doesn't tell you: 
1. Whether the chunk actually answers the question posed,
2. Whether the chunk is factually correct or up to date, 
3. Whether the match is semantically meaningful or just uses superfically similar wording. 

## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [40]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [41]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [42]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 15, score 0.657
17,120,121 This approach will
eliminate existing infections, as well as decrease the risk of further
infestation and subsequent associated clinical problems. Canine and
feline housemates may be at risk of transmission of infectious
parasites including roundworm and ﬂeas and therefore should be
treated in synchronicity with newly acquired kittens or cats. Pre-
venting cats ’ access to gardens and children ’s play sand areas will,
combined with parasite prophylaxis, decrease environmental con-
tamination with infectious and zoonotic agents such as hookworms
and Toxoplasma gondii.
33
Routine, regular use of broad-spectrum products is likely to be
beneﬁcial for the majority of pet cats, regardless of lifestyle. Certain
outdoor lifestyles, geographic location, and whether a cat spends time
away from the home (travel, boarding facilities, groomer, etc.) may
increase the existing risk of parasitic infection. Thus, recommen-
dations fo

In [43]:
answer_k = 4

result = answer_question(
    "What preventive care is recommended for cats?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Preventive care for cats in the guideline includes at least annual veterinary examinations for all cats, with more frequent visits based on individual needs [Source 4]. It also mentions client education on topics such as sterilization, claw care, identification and microchipping, and disaster preparedness [Source 4].

For parasite prevention, routine regular use of broad-spectrum products is likely beneficial for most pet cats, and cats with outdoor access, travel, boarding, or certain geographic risks may need additional prevention planning [Source 1]. Keeping cats away from gardens and children’s sand areas can also help reduce environmental contamination [Source 1].

If you want, I can also summarize preventive care by life stage.

Sources:
{'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1549, 'score': 0.6571027003607006}
{'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 2, 'start_index': 821, 'score': 0.6480204977

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [44]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
The guidelines recommend at least annual veterinary examinations for all cats, with more frequent visits based on individual needs; senior cats should be seen at least every 6 months [Source 4]. Preventive care also includes client education and discussion of topics such as sterilization, claw care, identification/microchipping, and disaster preparedness [Source 4]. In addition, regular broad-spectrum parasite prevention is likely beneficial for most pet cats, and prevention should be tailored to lifestyle and risk factors such as outdoor access or travel/boarding [Source 1].
Question: What symptoms should make me call a veterinarian?
You should contact a veterinarian if your cat has changes in appetite, increased drinking or urination, vomiting, vomiting hairballs, diarrhea, weight loss or other weight changes, chronic GI signs, increased nocturnal activity or vocalization, or changes in normal habits/activity. These can indicate

##Results
Question: What preventive care is recommended for cats?
Preventive care for cats in the guideline includes:

- **At least annual veterinary examinations for all cats**, with **more frequent visits as needed** based on individual risk and needs [Source 4].
- For **senior cats**, exams **at least every 6 months**, and more often if they have chronic conditions [Source 4].
- **Parasite prevention/prophylaxis**, with routine, regular use of **broad-spectrum products** likely beneficial for most pet cats [Source 1].
- **Risk-based parasite control**, since outdoor lifestyle, travel, boarding, and geography can increase infection risk [Source 1].
- **Treating housemates in synchronicity** when a newly acquired kitten or cat may carry infectious parasites like roundworm or fleas [Source 1].
- **Limiting access to gardens and children’s play sand areas** to reduce environmental contamination with parasites and zoonotic agents [Source 1].

If you want, I can also summarize the preventive care topics mentioned for all life stages.

====================================================================================================
Question: What symptoms should make me call a veterinarian?
Based on the guideline context, you should contact a veterinarian if your cat has:

- vomiting, including vomiting hairballs, especially if it is happening repeatedly or more often than usual [Source 1]
- diarrhea [Source 1]
- changes in appetite [Source 1]
- increased urination or drinking (polyuria/polydipsia) [Source 1]
- increased nocturnal activity or vocalization [Source 1][Source 3]
- changes in normal habits or activity [Source 1]
- changes in demeanor, activity level, or behavior over time [Source 4]
- reduced jumping or climbing, which may suggest pain, mobility issues, or muscle weakness [Source 2]
- any chronic gastrointestinal signs or other signs that could suggest early disease [Source 1]

If any of these symptoms are urgent or severe, contact a veterinarian promptly.

====================================================================================================
Question: What should I know about feeding a healthy adult cat?
For a healthy adult cat, feeding should be based on the cat’s life stage, body condition score (BCS), muscle condition score (MCS), activity level, neuter status, and overall health [Source 2][Source 3]. A good starting point is to calculate resting energy requirements (RER) using:

**RER = 30 × body weight in kg + 70**  
Then estimate daily energy requirements (DER) by multiplying RER by a needs factor; for **young, healthy adults**, the factor is **1** [Source 1].

Also, choose a diet labeled with an **AAFCO statement of nutritional adequacy** to help avoid nutrient deficiencies [Source 2]. Cats should not be fed raw or dehydrated non-sterilized foods [Source 2].

If your cat is overweight, don’t just feed less of a maintenance diet, because that can cause vitamin and mineral deficiencies; prescription weight-loss diets are indicated for obesity treatment [Source 1].

====================================================================================================
Question: Can my cat help me file my taxes?
I don't have enough information in the provided cat health guideline PDF to answer that.

====================================================================================================

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

For all queries, the retrieved context does seem relevant before generation. The retrieval was able to provide reasonable results, along with citations for the source provided. In addition, the final query resulted in the model stating that it doesn't have enough information in the provided pdf to answer. This suggests that the system prompt was considered for the final result. 


## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

### 🏗️ Activity Notes

- Setting changed: Chunk Size remained 1000, and overlap was reduced to 100. Changed k = 4, to k=10
- Before:
Recommended preventive care for cats includes:

- **At least annual veterinary examinations for all cats**, with more frequent visits based on individual needs; **senior cats should be seen at least every 6 months** [Source 4].
- **Routine, regular use of broad-spectrum parasite prevention products** for most pet cats, with risk-based adjustments depending on lifestyle and exposure [Source 1].
- **Treating housemates in sync** when a newly acquired kitten or cat may expose other cats to parasites like roundworm and fleas [Source 1].
- **Reducing environmental exposure** by limiting access to gardens and children’s play sand areas to help lower contamination with parasites and zoonotic agents [Source 1].

If you want, I can also summarize the key preventive care topics mentioned for all cat life stages.

Sources:
1. 'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1549, 'score': 0.6571027003607006
2. 'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 2, 'start_index': 821, 'score': 0.6479746171198265 
3. 'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'page': 19, 'start_index': 1659, 'score': 0.6267473099907146
4. 'source_label': 'Source 4', 'file': 'cat_health_guidelines.pdf', 'page': 6, 'start_index': 0, 'score': 0.6201210751870962
- After:

The guidelines recommend **at least annual preventive healthcare examinations for all cats**, with **more frequent visits based on the cat’s individual needs**; **senior cats should be seen at least every 6 months** and more often if they have chronic conditions [Source 4]. Preventive care should be **individualized** based on factors like the cat’s life stage, indoor/outdoor exposure, contact with other cats, travel/boarding, and human–cat interaction/zoonotic risk [Source 5]. It also includes reviewing **medical history, diet, nutrition, behavior, oral health, parasite prevention, and routine diagnostic testing as appropriate** [Source 1] [Source 3] [Source 7] [Source 8].

Sources:
1. 'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 6, 'start_index': 4500, 'score': 0.6561650292794825
2. 'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 16, 'start_index': 2685, 'score': 0.6491263908568956
3. 'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1781, 'score': 0.6357530699459537
4. 'source_label': 'Source 4', 'file': 'cat_health_guidelines.pdf', 'page': 6, 'start_index': 0, 'score': 0.6201210751870962
5. 'source_label': 'Source 5', 'file': 'cat_health_guidelines.pdf', 'page': 6, 'start_index': 3631, 'score': 0.6199173379337295
6. 'source_label': 'Source 6', 'file': 'cat_health_guidelines.pdf', 'page': 21, 'start_index': 4549, 'score': 0.6188284785128024
7. 'source_label': 'Source 7', 'file': 'cat_health_guidelines.pdf', 'page': 16, 'start_index': 1792, 'score': 0.6173319125466854
8. 'source_label': 'Source 8', 'file': 'cat_health_guidelines.pdf', 'page': 14, 'start_index': 3545, 'score': 0.6160581844867892
9. 'source_label': 'Source 9', 'file': 'cat_health_guidelines.pdf', 'page': 19, 'start_index': 1805, 'score': 0.6096040433045552
10. 'source_label': 'Source 10', 'file': 'cat_health_guidelines.pdf', 'page': 6, 'start_index': 1833, 'score': 0.6005057301320005
- Did retrieval improve? Why or why not?
In general, the retrieval was different based on the fact that it incorporated more information into the response. In the first response, only sources 1 and 4 were cited (pages 15 and 6). In the second response, those two sources were cited (both pages) along with additional information. The overall responses were similar with regards to annual examinations, but the second response focused more on the individual needs of the cat, and in may opinnion provided a better answer overall. 

### 🏗️ Activity Notes

- Setting changed: Chunk Size remained 1000, and overlap was reduced. Changed k = 4, to k=10
- Before:
Recommended preventive care for cats includes:

- **At least annual veterinary examinations for all cats**, with more frequent visits based on individual needs; **senior cats should be seen at least every 6 months** [Source 4].
- **Routine, regular use of broad-spectrum parasite prevention products** for most pet cats, with risk-based adjustments depending on lifestyle and exposure [Source 1].
- **Treating housemates in sync** when a newly acquired kitten or cat may expose other cats to parasites like roundworm and fleas [Source 1].
- **Reducing environmental exposure** by limiting access to gardens and children’s play sand areas to help lower contamination with parasites and zoonotic agents [Source 1].

If you want, I can also summarize the key preventive care topics mentioned for all cat life stages.

Sources:
- {'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1549, 'score': 0.6571027003607006}
- {'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 2, 'start_index': 821, 'score': 0.6479746171198265}
- {'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'page': 19, 'start_index': 1659, 'score': 0.6267473099907146}
- {'source_label': 'Source 4', 'file': 'cat_health_guidelines.pdf', 'page': 6, 'start_index': 0, 'score': 0.6201210751870962}
- After:
- Did retrieval improve? Why or why not?